# api-ops-smoke — 样本主轨（C0 / C0.5）

- **C0**：环境隔离初始化 + `TestClient` 打 `/` · `/health`
- **C0.5**：验收已有 `documents/sample/` 索引（**默认不重建**；需要时再开 `REBUILD_SAMPLE`）

样本索引路径：`Dataset/documents/sample/`（与 `full/` 分目录）。

## C0 — 环境隔离与骨架自检

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
STAGE12 = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
REPO = STAGE12.parent

# --- 环境隔离：清掉可能冲突的 app / 上游模块，避免与其它阶段 notebook 串包 ---
for name in ("config", "bootstrap", "resources", "app"):
    sys.modules.pop(name, None)
for key in list(sys.modules):
    if key == "app" or key.startswith("app."):
        sys.modules.pop(key, None)

for p in (str(REPO), str(STAGE12)):
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(STAGE12))

from app.bootstrap import bootstrap_paths
from app.bridge11 import reset_stage11_cache

reset_stage11_cache()
paths = bootstrap_paths(STAGE12)
print("repo", paths["root"])
print("stage12", paths["stage12"])
print("stage11", paths["stage11"])
print("sys.path[0]", sys.path[0])
assert Path(sys.path[0]).resolve() == paths["stage12"].resolve()

In [ ]:
from fastapi.testclient import TestClient

from app.main import app
from app.deps import get_session_store, get_qa_logger
from app.bridge11 import load_stage11

client = TestClient(app)
r = client.get("/")
print("GET /", r.status_code, r.json())
assert r.status_code == 200 and r.json()["code"] == 0
assert r.json()["data"]["stage"] == "12-0"

h = client.get("/health")
print("GET /health", h.status_code, h.json().get("code"))
assert h.status_code == 200 and h.json()["code"] == 0

s11 = load_stage11()
assert get_session_store() is s11["deps"].get_session_store()
assert get_qa_logger() is s11["deps"].get_qa_logger()
print("singletons OK (shared with stage11 /qa)")
print("MemorySessionStore", type(get_session_store()).__name__)
print("C0 PASS")

## C0.5 — 验收 documents/sample（默认不重建）

索引已在阶段 0 落盘。日常只 **status 校验**。
若需强制重建：将 `REBUILD_SAMPLE = True` 后重跑本格。

In [ ]:
from dataset_paths import (
    CHUNKS_SAMPLE_JSONL,
    DOCUMENTS_SAMPLE_DIR,
    DOCUMENTS_SAMPLE_SQLITE,
    SLIM_JSONL,
)
from app.documents_index import build_documents_index, status

# 默认关闭：样本索引已建好；仅在需要时改为 True
REBUILD_SAMPLE = False

print("slim", SLIM_JSONL.exists(), SLIM_JSONL)
print("chunks_sample", CHUNKS_SAMPLE_JSONL.exists(), CHUNKS_SAMPLE_JSONL)
print("sample_dir", DOCUMENTS_SAMPLE_DIR)
print("sqlite", DOCUMENTS_SAMPLE_SQLITE, "exists=", DOCUMENTS_SAMPLE_SQLITE.exists())

if REBUILD_SAMPLE:
    def _cb(p):
        if p.get("phase") in {"writing", "sample_complete", "completed", "finalizing"}:
            print(
                f"  lines={p.get('processed_lines')} rows={p.get('valid_rows')} "
                f"matched={p.get('matched_sample')}/{p.get('sample_target')} "
                f"phase={p.get('phase')}"
            )

    manifest = build_documents_index(
        "sample",
        batch_size=500,
        resume=False,
        progress_cb=_cb,
    )
    print("rebuilt", manifest.get("status"), "row_count", manifest.get("row_count"))
else:
    print("REBUILD_SAMPLE=False — skip build; verifying existing index")

st = status("sample")
print("status", {k: st[k] for k in ("row_count", "completed", "sqlite", "sqlite_exists")})
assert st["sqlite_exists"], f"missing {DOCUMENTS_SAMPLE_SQLITE}"
assert st["completed"] and st["row_count"] and st["row_count"] > 0
print("C0.5 PASS — documents/sample ready for stages 1–4")